In [6]:
pip install pybiomart gseapy networkx requests --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.0/693.0 kB 1.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [pybiomart]/6 [gseapy]s-cache]
Note: you may need to restart the kernel to use updated packages.


In [8]:
# Cell 1: Curate the REAL Schultz et al. 2025 human reference (Supplementary Data 4 = MOESM5)
import os
import pandas as pd
import numpy as np

human_dir = os.path.expanduser("~/FYP_Regeneration/data/human_fingertip/")
tables_dir = os.path.expanduser("~/FYP_Regeneration/results/tables/")
os.makedirs(tables_dir, exist_ok=True)

# The REAL file — Nature's generic MOESM naming, confirmed by its sheet name
raw_path = os.path.join(human_dir, "41536_2025_441_MOESM5_ESM.xlsx")

if not os.path.exists(raw_path):
    raise FileNotFoundError(
        f"Could not find {raw_path}. Make sure you've copied "
        f"41536_2025_441_MOESM5_ESM.xlsx into {human_dir}"
    )

raw = pd.read_excel(raw_path, sheet_name="60 Signif DEPs")
print("Real columns found:", raw.columns.tolist())
print(raw.head())

# Build the curated frame using the REAL column names confirmed above
pairwise_cols = [
    "Coagulation         vs. Hypergranulation",
    "Coagulation        vs.      Proliferation",
    "Coagulation      vs. Epithelialization",
    "Hypergranulation        vs.       Proliferation",
    "Hypergranulation vs. Epithelialization",
    "Proliferation         vs.        Epithelialization",
]

def parse_pval(v):
    """Handles values like '<0.0001' (string) mixed with real floats."""
    if isinstance(v, str):
        v = v.strip()
        if v.startswith("<"):
            return float(v[1:])   # treat '<0.0001' as 0.0001 (a conservative upper bound)
        try:
            return float(v)
        except ValueError:
            return np.nan
    return v

curated = pd.DataFrame({
    "accession_number": raw["Accession Number"].astype(str).str.split(" \[").str[0],  # strip "[9]" cluster-count suffixes
    "gene_symbol": raw["Gene"].astype(str).str.strip().str.upper(),
    "protein_description": raw["Protein Description"],
    "molecular_weight": raw["Molecular Weight"],
})

for col in pairwise_cols:
    clean_name = "p_" + col.split("vs.")[0].strip().lower() + "_vs_" + col.split("vs.")[1].strip().lower().replace(" ", "")
    curated[clean_name] = raw[col].apply(parse_pval)

curated["significant_any_comparison"] = raw["Significant for 1 or more pairwise comparisons? "]

# Check for missing/duplicate gene symbols before this becomes your reference set
n_missing = curated["gene_symbol"].isna().sum()
n_dupes = curated["gene_symbol"].duplicated().sum()
print(f"\nMissing gene symbols: {n_missing}")
print(f"Duplicate gene symbols: {n_dupes}")
if n_dupes > 0:
    print(curated[curated['gene_symbol'].duplicated(keep=False)].sort_values('gene_symbol'))

print(f"\nCurated human reference: {len(curated)} proteins")
print(curated.head())

out_path = os.path.join(tables_dir, "human_fingertip_curated_reference.csv")
curated.to_csv(out_path, index=False)
print(f"Saved real curated reference to: {out_path}")

Real columns found: ['Accession Number', 'Gene', 'Accession Number.1', 'Protein Description', 'Molecular Weight', 'Coagulation         vs. Hypergranulation', 'Coagulation        vs.      Proliferation', 'Coagulation      vs. Epithelialization', 'Hypergranulation        vs.       Proliferation', 'Hypergranulation vs. Epithelialization', 'Proliferation         vs.        Epithelialization', 'Significant for 1 or more pairwise comparisons? ']
  Accession Number   Gene Accession Number.1  \
0           P01023    A2M             P01023   
1       P60709 [9]   ACTB         P60709 [9]   
2       P12814 [4]  ACTN1         P12814 [4]   
3           P02768    ALB             P02768   
4       P16157 [3]   ANK1         P16157 [3]   

                                 Protein Description Molecular Weight  \
0  Alpha-2-macroglobulin OS=Homo sapiens OX=9606 ...          163 kDa   
1  Cluster of Actin, cytoplasmic 1 OS=Homo sapien...           42 kDa   
2  Cluster of Alpha-actinin-1 OS=Homo sapiens OX

In [9]:
# Cell 2: Real ortholog mapping and cross-species comparison
import os
import pandas as pd
import matplotlib.pyplot as plt

tables_dir = os.path.expanduser("~/FYP_Regeneration/results/tables/")
fig_dir = os.path.expanduser("~/FYP_Regeneration/results/figures/cross_species/")
os.makedirs(fig_dir, exist_ok=True)

# 1. Load your REAL significant gene sets
axolotl_df = pd.read_csv(os.path.join(tables_dir, "axolotl_de_markers_sig.csv"))
mouse_df = pd.read_csv(os.path.join(tables_dir, "mouse_de_genes_sig.csv"))
human_df = pd.read_csv(os.path.join(tables_dir, "human_fingertip_curated_reference.csv"))

raw_axolotl = set(axolotl_df["gene"].dropna().astype(str).str.upper())
raw_mouse = set(mouse_df["gene"].dropna().astype(str).str.upper())
raw_human = set(human_df["gene_symbol"].dropna().astype(str).str.upper())

print(f"Real gene set sizes going into mapping: axolotl={len(raw_axolotl)}, "
      f"mouse={len(raw_mouse)}, human={len(raw_human)}")

# 2. Mouse -> human ortholog mapping via real Ensembl BioMart query
from pybiomart import Server

print("\nQuerying Ensembl BioMart for mouse->human orthologs (requires internet)...")
server = Server(host="http://www.ensembl.org")
mouse_dataset = server.marts["ENSEMBL_MART_ENSEMBL"].datasets["mmusculus_gene_ensembl"]
homology = mouse_dataset.query(
    attributes=["external_gene_name", "hsapiens_homolog_associated_gene_name"]
)
homology.columns = ["mouse_symbol", "human_ortholog"]
homology = homology.dropna()
homology["mouse_symbol"] = homology["mouse_symbol"].str.upper()
homology["human_ortholog"] = homology["human_ortholog"].str.upper()
mouse_to_human = dict(zip(homology["mouse_symbol"], homology["human_ortholog"]))

mouse_ortho = set(mouse_to_human[g] for g in raw_mouse if g in mouse_to_human)
n_unmapped_mouse = len(raw_mouse) - len(mouse_ortho)
print(f"Mouse genes successfully mapped to a human ortholog: {len(mouse_ortho)} "
      f"({n_unmapped_mouse} had no BioMart-listed human ortholog and were dropped)")

# 3. Axolotl and human: no independent programmatic ortholog database is available in this
# environment (Ambystoma mexicanum is not in Ensembl BioMart). The axolotl gene models you're
# using were themselves annotated via homology to known vertebrate genes (this is why you saw
# real symbols like COL1A1, HOXA2 come out of Step 2) — so we compare gene-symbol identity
# directly. THIS IS A DOCUMENTED METHODOLOGICAL LIMITATION, not a full independent ortholog
# call — say so explicitly in your methods and results sections.
axolotl_ortho = raw_axolotl          # symbol-identity comparison, not a verified ortholog call
human_ortho = raw_human               # already human gene symbols

print(f"\nFinal comparable sets: axolotl={len(axolotl_ortho)}, "
      f"mouse (mapped to human)={len(mouse_ortho)}, human={len(human_ortho)}")

# 4. Real overlaps — nothing added, nothing forced
axolotl_mouse = axolotl_ortho & mouse_ortho
axolotl_human = axolotl_ortho & human_ortho
mouse_human = mouse_ortho & human_ortho
tri_conserved = axolotl_ortho & mouse_ortho & human_ortho

print(f"\nCross-Species Overlaps (REAL, computed):")
print(f"- Axolotl & Mouse: {len(axolotl_mouse)}")
print(f"- Axolotl & Human: {len(axolotl_human)}")
print(f"- Mouse & Human: {len(mouse_human)}")
print(f"- Tri-Species Conserved: {len(tri_conserved)}")
if len(tri_conserved) == 0:
    print("Zero tri-species conserved genes is a legitimate, reportable result — "
          "it does not mean the analysis failed.")
print("Tri-species conserved genes:", sorted(tri_conserved))

# 5. Build the real integrated table
all_genes = sorted(axolotl_ortho | mouse_ortho | human_ortho)
rows = []
for gene in all_genes:
    in_ax, in_ms, in_hu = gene in axolotl_ortho, gene in mouse_ortho, gene in human_ortho
    rows.append({
        "Ortholog_Gene_Symbol": gene,
        "Axolotl_scRNA": in_ax,
        "Mouse_BulkRNA": in_ms,
        "Human_Proteomics": in_hu,
        "Conservation_Score": sum([in_ax, in_ms, in_hu]),
    })
integrated_df = pd.DataFrame(rows)
conserved_df = integrated_df[integrated_df["Conservation_Score"] >= 2].sort_values(
    "Conservation_Score", ascending=False
)

integrated_df.to_csv(os.path.join(tables_dir, "tri_species_orthologs_all.csv"), index=False)
conserved_df.to_csv(os.path.join(tables_dir, "tri_species_conserved_markers.csv"), index=False)

# 6. Real summary plot
plt.figure(figsize=(9, 5))
categories = ["Axolotl & Mouse", "Axolotl & Human", "Mouse & Human", "Tri-Species"]
counts = [len(axolotl_mouse), len(axolotl_human), len(mouse_human), len(tri_conserved)]
bars = plt.bar(categories, counts, color=["#2b5c8f", "#d95f02", "#7570b3", "#e7298a"])
plt.ylabel("Shared Gene Count")
plt.title("Cross-Species Ortholog Overlap (real computed values)")
for bar in bars:
    y = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, y + 0.1, int(y), ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "cross_species_ortholog_summary.png"), dpi=150)
plt.close()

print("\nStep 5 & 6 complete — results reflect your real data, whatever they turned out to be.")

Real gene set sizes going into mapping: axolotl=8532, mouse=196, human=60

Querying Ensembl BioMart for mouse->human orthologs (requires internet)...
Mouse genes successfully mapped to a human ortholog: 149 (47 had no BioMart-listed human ortholog and were dropped)

Final comparable sets: axolotl=8532, mouse (mapped to human)=149, human=60

Cross-Species Overlaps (REAL, computed):
- Axolotl & Mouse: 37
- Axolotl & Human: 17
- Mouse & Human: 0
- Tri-Species Conserved: 0
Zero tri-species conserved genes is a legitimate, reportable result — it does not mean the analysis failed.
Tri-species conserved genes: []

Step 5 & 6 complete — results reflect your real data, whatever they turned out to be.


In [10]:
# Cell 3: Real pathway enrichment via gseapy/Enrichr on the REAL conserved gene list
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gseapy as gp

tables_dir = os.path.expanduser("~/FYP_Regeneration/results/tables/")
fig_dir = os.path.expanduser("~/FYP_Regeneration/results/figures/cross_species/")

conserved_df = pd.read_csv(os.path.join(tables_dir, "tri_species_conserved_markers.csv"))
genes_list = conserved_df.loc[conserved_df["Conservation_Score"] == 3, "Ortholog_Gene_Symbol"].tolist()

print(f"Real tri-species conserved gene count: {len(genes_list)}")
print(genes_list)

if len(genes_list) < 3:
    print("\nFewer than 3 conserved genes — pathway enrichment on this few genes is "
          "statistically unreliable. Falling back to Conservation_Score >= 2.")
    genes_list = conserved_df.loc[conserved_df["Conservation_Score"] >= 2, "Ortholog_Gene_Symbol"].tolist()
    print(f"Falling back to Conservation_Score >= 2: {len(genes_list)} genes -> {genes_list}")

if len(genes_list) == 0:
    print("No genes to enrich — skipping. This is a real, reportable outcome, not an error.")
    enrichment_df = pd.DataFrame()
else:
    enr = gp.enrichr(
        gene_list=genes_list,
        gene_sets=["GO_Biological_Process_2023", "KEGG_2021_Human", "Reactome_2022"],
        organism="human",
        outdir=None,
    )
    enrichment_df = enr.results.sort_values("Adjusted P-value").reset_index(drop=True)
    enrichment_df.to_csv(os.path.join(tables_dir, "pathway_enrichment_results.csv"), index=False)
    print(f"\nReal enrichment results: {len(enrichment_df)} terms returned")
    print(enrichment_df[["Term", "Gene_set", "Adjusted P-value", "Genes"]].head(15))

    top = enrichment_df.head(15).copy()
    top["-log10_padj"] = -np.log10(top["Adjusted P-value"].clip(lower=1e-300))
    top["gene_count"] = top["Genes"].apply(lambda x: len(x.split(";")))

    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(
        top["-log10_padj"], top["Term"],
        s=top["gene_count"] * 60, c=top["Combined Score"], cmap="viridis",
        alpha=0.85, edgecolors="black",
    )
    plt.colorbar(scatter, label="Combined Score")
    plt.xlabel("-log10 Adjusted P-value")
    plt.title("Pathway Enrichment — Real Conserved Gene Set")
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "pathway_enrichment_dotplot.png"), dpi=150)
    plt.close()

print("\nStep 7 complete.")

Real tri-species conserved gene count: 0
[]

Fewer than 3 conserved genes — pathway enrichment on this few genes is statistically unreliable. Falling back to Conservation_Score >= 2.
Falling back to Conservation_Score >= 2: 54 genes -> ['A2M', 'ABAT', 'ACTB', 'ACTN1', 'ADTRP', 'APLP1', 'ATP12A', 'BAMBI', 'C1QTNF2', 'C3', 'CAT', 'CCBE1', 'CDH6', 'CFB', 'CFH', 'COX4I2', 'CPN1', 'CYBRD1', 'DSP', 'FBN2', 'FGF10', 'FN1', 'GC', 'HBB', 'HTRA4', 'IFITM3', 'IGFBP5', 'IL11', 'IL1B', 'ITGBL1', 'LEF1', 'LRRC15', 'MAL', 'MATN4', 'MGST1', 'MSI1', 'MSX2', 'MYH9', 'OSR2', 'PAMR1', 'PAX9', 'PRDX2', 'PRTG', 'PZP', 'RNASET2', 'SERPINA1', 'SERPINB1', 'SMOC1', 'SOX11', 'TCF7', 'TF', 'THBS4', 'TWIST2', 'VWA5A']

Real enrichment results: 973 terms returned
                                                 Term  \
0                 Complement and coagulation cascades   
1   Regulation Of Osteoblast Differentiation (GO:0...   
2   Positive Regulation Of Cell Population Prolife...   
3   Post-translational Prote

In [11]:
# Cell 4: Real STRING-db network query for the REAL conserved gene list
import os
import pandas as pd
import requests
import networkx as nx
import matplotlib.pyplot as plt

tables_dir = os.path.expanduser("~/FYP_Regeneration/results/tables/")
fig_dir = os.path.expanduser("~/FYP_Regeneration/results/figures/cross_species/")

conserved_df = pd.read_csv(os.path.join(tables_dir, "tri_species_conserved_markers.csv"))
genes = conserved_df.loc[conserved_df["Conservation_Score"] >= 2, "Ortholog_Gene_Symbol"].tolist()
print(f"Querying STRING for {len(genes)} real genes: {genes}")

if len(genes) < 2:
    print("Fewer than 2 genes — no network to build. This is a real result, not an error.")
    nodes_df, edges_df = pd.DataFrame(), pd.DataFrame()
else:
    string_api_url = "https://string-db.org/api"
    params = {
        "identifiers": "%0d".join(genes),
        "species": 9606,
        "required_score": 700,
        "caller_identity": "fyp_regeneration_project",
    }
    response = requests.post(f"{string_api_url}/tsv-no-header/network", data=params)
    response.raise_for_status()

    edges = []
    for line in response.text.strip().split("\n"):
        if not line:
            continue
        parts = line.split("\t")
        # STRING network TSV columns (fixed order):
        # 0 stringId_A, 1 stringId_B, 2 preferredName_A, 3 preferredName_B,
        # 4 ncbiTaxonId, 5 score (COMBINED score — this is the one to use),
        # 6 nscore, 7 fscore, 8 pscore, 9 ascore, 10 escore, 11 dscore, 12 tscore
        edges.append((parts[2], parts[3], float(parts[5])))   # <-- fixed: was parts[-1]

    edges_df = pd.DataFrame(edges, columns=["Source", "Target", "Combined_Score"]).drop_duplicates()
    print(f"Real STRING interactions returned: {len(edges_df)}")

    G = nx.Graph()
    G.add_nodes_from(genes)
    for u, v, w in edges_df.itertuples(index=False):
        G.add_edge(u, v, weight=w)

    degree_dict = dict(G.degree())
    node_stats = [{
        "Gene_Symbol": g,
        "Degree": degree_dict.get(g, 0),
        "Degree_Centrality": round(nx.degree_centrality(G).get(g, 0), 3),
        "Betweenness_Centrality": round(nx.betweenness_centrality(G).get(g, 0), 3),
        "Closeness_Centrality": round(nx.closeness_centrality(G).get(g, 0), 3),
    } for g in genes]
    nodes_df = pd.DataFrame(node_stats).sort_values("Degree", ascending=False)

    nodes_df.to_csv(os.path.join(tables_dir, "ppi_network_nodes.csv"), index=False)
    edges_df.to_csv(os.path.join(tables_dir, "ppi_network_edges.csv"), index=False)

    if len(edges_df) > 0:
        plt.figure(figsize=(9, 7))
        pos = nx.spring_layout(G, seed=42, k=0.85)
        sizes = [degree_dict.get(n, 0) * 500 + 100 for n in G.nodes()]
        colors = [degree_dict.get(n, 0) for n in G.nodes()]
        nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=colors, cmap=plt.cm.YlOrRd, edgecolors="black")
        nx.draw_networkx_edges(G, pos, width=1.5, alpha=0.5, edge_color="darkgray")
        nx.draw_networkx_labels(G, pos, font_size=10, font_weight="bold")
        plt.title("Real STRING PPI Network — Conserved Regeneration Genes")
        plt.axis("off")
        plt.tight_layout()
        plt.savefig(os.path.join(fig_dir, "string_ppi_network.png"), dpi=150)
        plt.close()
    else:
        print("STRING returned zero interactions among these genes at this confidence threshold.")

print("\nStep 8 complete.")

Querying STRING for 54 real genes: ['A2M', 'ABAT', 'ACTB', 'ACTN1', 'ADTRP', 'APLP1', 'ATP12A', 'BAMBI', 'C1QTNF2', 'C3', 'CAT', 'CCBE1', 'CDH6', 'CFB', 'CFH', 'COX4I2', 'CPN1', 'CYBRD1', 'DSP', 'FBN2', 'FGF10', 'FN1', 'GC', 'HBB', 'HTRA4', 'IFITM3', 'IGFBP5', 'IL11', 'IL1B', 'ITGBL1', 'LEF1', 'LRRC15', 'MAL', 'MATN4', 'MGST1', 'MSI1', 'MSX2', 'MYH9', 'OSR2', 'PAMR1', 'PAX9', 'PRDX2', 'PRTG', 'PZP', 'RNASET2', 'SERPINA1', 'SERPINB1', 'SMOC1', 'SOX11', 'TCF7', 'TF', 'THBS4', 'TWIST2', 'VWA5A']
Real STRING interactions returned: 28

Step 8 complete.
